### **Representaciones distribuidas**

El objetivo de este cuaderno, es que el estudiante debería poder:

- explicar qué es una representación distribuida y por qué supera a las representaciones discretas,
- distinguir entre embeddings estáticos y contextuales,
- comprender la intuición matemática de **NNLM**, **C&W**, **CBOW**, **Skip-gram** y métodos subpalabra,
- construir representaciones simples de frases y oraciones,
- conectar estas ideas con el problema de **alineamiento visual-semántico** que aparece en aprendizaje multimodal.

#### **1. Representaciones distribuidas: idea general**

Una representación distribuida asigna a cada unidad lingüística, palabra, frase, oración o documento,  un vector denso en un espacio continuo. La hipótesis central es que **unidades que aparecen en contextos similares tienden a tener significados similares**.

Frente a una codificación *one-hot*:
- la dimensión deja de crecer con el tamaño del vocabulario,
- la similitud semántica puede medirse geométricamente,
- se favorece la generalización,
- se prepara el camino para modelos multimodales, donde texto e imagen también deben proyectarse a espacios comparables.

Una analogía clásica que popularizó Word2Vec es:

![](https://jalammar.github.io/images/word2vec/king-white-embedding.png)

![](https://jalammar.github.io/images/word2vec/king-colored-embedding.png)

![](https://jalammar.github.io/images/word2vec/king-man-woman-embedding.png)


![](https://jalammar.github.io/images/word2vec/king-analogy-viz.png)

La igualdad no debe interpretarse literalmente, sino como una regularidad geométrica aproximada del espacio vectorial.

#### **2. Embeddings de palabras**

Los **embeddings de palabras** son representaciones vectoriales densas de dimensión fija que buscan capturar regularidades **semánticas**, **sintácticas** y, en algunos casos, **morfológicas** o **contextuales** del lenguaje.

![](word_embedding.png)

Si $e_w \in \mathbb{R}^d$ denota el embedding de una palabra $w$, una medida estándar de cercanía entre dos palabras es la **similitud coseno**:

$$
\operatorname{sim}(w_i,w_j)=
\frac{e_{w_i}^{\top}e_{w_j}}{\lVert e_{w_i}\rVert\,\lVert e_{w_j}\rVert}.
$$

La idea central es que palabras que aparecen en **contextos similares** tienden a tener vectores cercanos.  
Esta intuición se relaciona con la **hipótesis distribucional**: *una palabra se caracteriza, en parte, por los contextos en los que aparece*.

##### **2.1. ¿Por qué embeddings y no representaciones discretas?**

Representaciones como **one-hot encoding** o **bag of words** tienen varias limitaciones:

- son de **alta dimensión** y muy dispersas,
- no capturan similitud entre palabras,
- no generalizan bien entre términos semánticamente cercanos,
- tratan cada palabra como una unidad aislada.

Los embeddings resuelven parcialmente estos problemas porque:
- reducen la dimensión,
- permiten medir similitud,
- facilitan la generalización,
- y pueden incorporarse como parámetros entrenables dentro de modelos neuronales.

##### **2.2. Principales variantes de embeddings**

**a. Embeddings basados en conteos**
Se construyen a partir de matrices de coocurrencia palabra-contexto y luego se ponderan o reducen dimensionalmente.

Ejemplos:
- matriz término-contexto,
- **PPMI**,
- **PPMI + SVD**,
- **LSA/Latent Semantic Analysis**.

Ventaja:
- interpretación relativamente clara desde semántica distribucional clásica.

Limitación:
- suelen depender de preprocesamiento cuidadoso y no siempre modelan bien relaciones no lineales.

**b. Embeddings predictivos**

En lugar de factorizar una matriz explícita, aprenden vectores optimizando una tarea predictiva.

Ejemplos:
- **Neural Language Model (NNLM)**,
- **Word2Vec-CBOW**,
- **Word2Vec-Skip-gram**.

En estos modelos, el embedding se aprende como parte de una función objetivo que predice:
- una palabra a partir de su contexto, o
- el contexto a partir de una palabra.

Ventaja:
- suelen producir embeddings más compactos y efectivos en muchas tareas.

**c. Embeddings preentrenados**
Son vectores aprendidos previamente en grandes corpus y luego reutilizados.

Ejemplos:
- **Word2Vec**,
- **GloVe**,
- **fastText**.

En este caso, una palabra $w$ ya tiene asociado un vector aprendido sobre un corpus externo:
$$
w \mapsto e_w.
$$

Ventajas:
- aprovechan grandes cantidades de texto,
- reducen costo de entrenamiento,
- mejoran el desempeño cuando hay pocos datos etiquetados.

**d. Embeddings subléxicos**
Incorporan información interna de la palabra: prefijos, sufijos, raíces o n-gramas de caracteres.

Ejemplo principal:
- **fastText**.

La idea es representar una palabra no solo como una unidad atómica, sino como composición de subunidades:

$$
e_w \approx \sum_{g \in G(w)} e_g
$$

donde $G(w)$ es el conjunto de n-gramas de caracteres de la palabra $w$.

Ventajas:
- manejan mejor palabras raras,
- ayudan con variación morfológica,
- son útiles en lenguas con morfología rica.

**e. Embeddings contextuales**
A diferencia de los embeddings estáticos, aquí una misma palabra puede tener **distintos vectores según el contexto**.

Ejemplo:
- la palabra *bank* no debería representarse igual en:
  - *river bank*,
  - *bank account*.

Formalmente, en lugar de un solo vector $e_w$, se obtiene una representación dependiente del contexto:

$$
e_w^{(c)} = f(w, c)
$$

donde $c$ es el contexto lingüístico.

Ejemplos:
- **ELMo**,
- **BERT**,
- representaciones derivadas de transformers.

Ventaja:
- capturan polisemia y desambiguación contextual.

**f. Embeddings ajustados a tarea**
En muchos sistemas modernos, los embeddings:
- pueden inicializarse aleatoriamente,
- cargarse desde un modelo preentrenado,
- y luego **ajustarse** durante el entrenamiento de una tarea específica.

Es decir, si $e_w^{(0)}$ es el embedding inicial, durante el entrenamiento puede evolucionar a:

$$
e_w^{(t+1)} = e_w^{(t)} - \eta \nabla_{e_w}\mathcal{L}
$$

donde $\mathcal{L}$ es la función de pérdida de la tarea y $\eta$ la tasa de aprendizaje.

Esto permite adaptar los embeddings al dominio o problema concreto.

##### **2.3. Geometría semántica en el espacio de embeddings**

Una vez representadas las palabras como vectores, podemos estudiar relaciones geométricas:

- cercanía semántica,
- agrupamientos léxicos,
- analogías,
- regularidades sintácticas.

En algunos embeddings estáticos clásicos, aparecen patrones del tipo:

$$
e_{\text{king}} - e_{\text{man}} + e_{\text{woman}} \approx e_{\text{queen}}
$$

Este tipo de comportamiento no debe interpretarse como “comprensión” plena del lenguaje, pero sí como evidencia de que ciertas regularidades se reflejan geométricamente en el espacio vectorial.

##### **2.4. Limitaciones de los embeddings de palabras**

Aunque fueron un gran avance, los embeddings de palabras también tienen limitaciones:

- pueden heredar sesgos del corpus,
- los embeddings estáticos no modelan bien polisemia,
- la similitud geométrica no siempre equivale a similitud semántica profunda,
- no capturan por sí solos estructura discursiva ni composicionalidad completa.

Por eso, históricamente el campo evolucionó desde:
- matrices de coocurrencia,
- embeddings estáticos,
- embeddings subléxicos,
- hasta embeddings contextuales y modelos de lenguaje más complejos.

##### **2.5. Relevancia para aprendizaje multimodal**

Esta sección es importante para el curso porque, en aprendizaje multimodal temprano, el texto también debía representarse como vector.  
Los primeros modelos de alineamiento visual-semántico no partían de texto “crudo”, sino de **representaciones distribuidas del lenguaje** que luego se alineaban con representaciones visuales.

En otras palabras, antes de aprender un espacio compartido imagen-texto, primero hay que responder:

$$
x_{\text{texto}} \longrightarrow e_{\text{texto}}
$$

y solo después pensar en cómo alinear ese vector con una representación visual:

$$
e_{\text{texto}} \leftrightarrow e_{\text{imagen}}.
$$

In [ ]:
import numpy as np

# Vectores de ejemplo para similitud coseno
emb = {
    "king":   np.array([0.8, 0.7, 0.2]),
    "queen":  np.array([0.82, 0.68, 0.22]),
    "apple":  np.array([0.1, 0.2, 0.9]),
}

def cosine(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

print("sim(king, queen) =", round(cosine(emb["king"], emb["queen"]), 4))
print("sim(king, apple) =", round(cosine(emb["king"], emb["apple"]), 4))

#### **3. Modelos clásicos de embeddings**

#### **3.1 Neural Network Language Model (NNLM)**

El **NNLM** propuesto por [Bengio](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf) aprende simultáneamente:

1. una representación distribuida de las palabras (**embeddings**), y  
2. un modelo de lenguaje que estima la probabilidad de la siguiente palabra dada una ventana de contexto fija.

El objetivo es modelar:

$$
P(w_t \mid w_{t-n+1}, \dots, w_{t-1})
$$

donde el contexto está formado por las $n-1$ palabras anteriores.

##### **Paso 1. Representación de cada palabra**

Cada palabra $w$ del vocabulario $V$ puede representarse inicialmente mediante un vector one-hot $x_w \in \mathbb{R}^{|V|}$.  
Luego, una matriz de embeddings $C \in \mathbb{R}^{|V| \times d}$ proyecta esa representación discreta a un espacio continuo de dimensión $d$:

$$
e_w = C^\top x_w
$$

Como $x_w$ es one-hot, esta operación equivale a seleccionar la fila correspondiente de la matriz $C$.  
Así, cada palabra del contexto se transforma en un embedding denso:

$$
e_{w_{t-n+1}}, \dots, e_{w_{t-1}}
$$

con cada $e_w \in \mathbb{R}^d$.

##### **Paso 2. Concatenación de embeddings del contexto**

El NNLM original no promedia embeddings, sino que los **concatena** para preservar la posición de cada palabra en el contexto:

$$
x = [e_{w_{t-n+1}}, \dots, e_{w_{t-1}}]
$$

Si cada embedding tiene dimensión $d$, entonces:

$$
x \in \mathbb{R}^{(n-1)d}
$$

Esta concatenación permite que el modelo distinga entre las posiciones del contexto.

##### **Paso 3. Capa oculta**

El vector concatenado pasa a una capa oculta con activación no lineal:

$$
h = f\!\left(W [e_{w_{t-n+1}}, \dots, e_{w_{t-1}}] + b_h\right)
$$

donde:

- $W \in \mathbb{R}^{m \times (n-1)d}$ es la matriz de pesos,
- $b_h \in \mathbb{R}^{m}$ es el sesgo,
- $f(\cdot)$ es una función no lineal, por ejemplo $\tanh$ o sigmoide,
- $h \in \mathbb{R}^{m}$ es la representación oculta del contexto.

Esta capa aprende una representación interna del contexto que resume la información relevante para predecir la siguiente palabra.


##### **Paso 4. Puntajes para cada palabra del vocabulario**

A partir de $h$, el modelo calcula un puntaje para cada palabra del vocabulario mediante una transformación lineal:

$$
z = Uh + b
$$

donde:

- $U \in \mathbb{R}^{|V| \times m}$,
- $b \in \mathbb{R}^{|V|}$,
- $z \in \mathbb{R}^{|V|}$.

Cada componente $z_w$ representa el puntaje asociado a la palabra $w$ como posible siguiente palabra.

En particular:

$$
z_w = u_w^\top h + b_w
$$

donde $u_w^\top$ es la fila de $U$ correspondiente a la palabra $w$.

##### **Paso 5. Softmax para obtener probabilidades**

Los puntajes se convierten en probabilidades usando la función **softmax**:

$$
P(w_t = w \mid w_{t-n+1}, \dots, w_{t-1})
=
\frac{\exp(z_w)}
{\sum_{w' \in V} \exp(z_{w'})}
$$

Sustituyendo $z = Uh + b$, se obtiene:

$$
P(w_t \mid w_{t-n+1}, \dots, w_{t-1})
=
\frac{\exp\!\big((Uh+b)_{w_t}\big)}
{\sum_{w \in V} \exp\!\big((Uh+b)_w\big)}
$$

Esta es la ecuación estándar del NNLM.


##### **Interpretación de la ecuación**

La ecuación sale de encadenar estas operaciones:

$$
\text{contexto}
\rightarrow
\text{embeddings}
\rightarrow
\text{concatenación}
\rightarrow
\text{capa oculta } h
\rightarrow
\text{puntajes } Uh+b
\rightarrow
\text{softmax}
\rightarrow
P(w_t \mid \text{contexto})
$$

Es decir:

1. cada palabra del contexto se convierte en un embedding,
2. los embeddings se concatenan,
3. una red neuronal transforma el contexto en una representación oculta,
4. esa representación se proyecta al espacio del vocabulario,
5. softmax convierte los puntajes en probabilidades.

##### **Función de pérdida**

Durante el entrenamiento, se maximiza la probabilidad de la palabra observada en el corpus, o equivalentemente se minimiza la pérdida de entropía cruzada negativa:

$$
\mathcal{L}
=
-\sum_t \log P(w_t \mid w_{t-n+1}, \dots, w_{t-1})
$$

Los parámetros que se aprenden son:

$$
\theta = \{C, W, b_h, U, b\}
$$

donde:

- $C$ aprende los embeddings,
- $W$ y $b_h$ aprenden la representación oculta del contexto,
- $U$ y $b$ producen la distribución de salida sobre el vocabulario.

##### **Importancia del NNLM**

La contribución clave del NNLM es que palabras semánticamente parecidas tienden a tener embeddings cercanos.  
Por ello, el modelo puede generalizar mejor que un modelo clásico de $n$-gramas basado únicamente en conteos.

Por ejemplo, si *gato* y *perro* tienen embeddings similares, el modelo puede transferir conocimiento entre contextos similares sin haber visto exactamente la misma secuencia en el entrenamiento.

##### **Limitación principal**

El costo computacional de la salida es alto porque el denominador del softmax requiere recorrer todo el vocabulario:

$$
\sum_{w \in V} \exp\!\big((Uh+b)_w\big)
$$

Cuando $|V|$ es muy grande, esta operación se vuelve costosa.  
Esa fue una de las razones por las que después aparecieron variantes más eficientes, como **CBOW**, **Skip-gram**, **hierarchical softmax** y **negative sampling**.

In [ ]:
import torch
import torch.nn as nn

vocab = ["I", "like", "dogs", "cats", "and"]
word2idx = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
d = 10
context_size = 2
hidden_dim = 20

class NNLM(nn.Module):
    def __init__(self, vocab_size, emb_dim, context_size, hidden_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.linear1 = nn.Linear(context_size * emb_dim, hidden_dim)
        self.tanh = nn.Tanh()
        self.linear2 = nn.Linear(hidden_dim, vocab_size)

    def forward(self, context_idxs):
        emb = self.emb(context_idxs).view(context_idxs.size(0), -1)
        h = self.tanh(self.linear1(emb))
        logits = self.linear2(h)
        return torch.softmax(logits, dim=-1)

model = NNLM(V, d, context_size, hidden_dim)
x = torch.tensor([[word2idx["I"], word2idx["like"]]])
print(model(x))

##### **3.2 C&W (Collobert & Weston)**

El modelo de **Collobert & Weston (C\&W)** propone una idea distinta al NNLM clásico de Bengio.  
En lugar de modelar explícitamente la probabilidad de la siguiente palabra mediante una distribución softmax sobre todo el vocabulario, C\&W aprende una función de **puntuación** que asigna valores altos a secuencias lingüísticamente plausibles y valores bajos a secuencias corruptas.

La idea central es:

- una ventana real de palabras $x^+$ debe recibir una puntuación alta,
- una ventana corrupta $x^-$ debe recibir una puntuación más baja.

Por tanto, el modelo no aprende una distribución de probabilidad explícita, sino una **función de ranking**.

###### **Ventana de contexto**

Supongamos una ventana de tamaño fijo compuesta por $m$ palabras:

$$
x = (w_1, w_2, \dots, w_m)
$$

Cada palabra $w_i$ se representa mediante un embedding:

$$
e_{w_i} \in \mathbb{R}^d
$$

Luego, los embeddings de la ventana se concatenan para formar un solo vector de entrada:

$$
z = [e_{w_1}, e_{w_2}, \dots, e_{w_m}] \in \mathbb{R}^{md}
$$

Esta concatenación preserva la posición de las palabras dentro de la ventana.

###### **Función de puntuación**

El modelo C\&W define una red neuronal que toma como entrada la ventana representada por embeddings concatenados y produce una **puntuación escalar**:

$$
s(x) \in \mathbb{R}
$$

Una formulación típica es:

$$
h = f(W z + b_h)
$$

$$
s(x) = u^\top h + b_s
$$

donde:

- $z \in \mathbb{R}^{md}$ es la concatenación de embeddings,
- $W \in \mathbb{R}^{p \times md}$ es la matriz de pesos de la capa oculta,
- $b_h \in \mathbb{R}^{p}$ es el sesgo de la capa oculta,
- $f(\cdot)$ es una no linealidad, por ejemplo $\tanh$,
- $h \in \mathbb{R}^{p}$ es la representación interna de la ventana,
- $u \in \mathbb{R}^{p}$ y $b_s \in \mathbb{R}$ producen la puntuación final.

Así, $s(x)$ mide cuán "bien formada" o plausible considera la red a la secuencia $x$.

###### **Ventana positiva y ventana negativa**

Se construyen dos tipos de ejemplos:

**Ventana positiva**  
Es una secuencia real extraída del corpus:

$$
x^+ = (w_1, w_2, \dots, w_m)
$$

**Ventana negativa**  
Se obtiene corrompiendo la ventana positiva, normalmente reemplazando una de las palabras (a menudo la palabra central) por otra palabra aleatoria del vocabulario:

$$
x^- = (w_1, \dots, \tilde{w}_j, \dots, w_m)
$$

donde $\tilde{w}_j$ es una palabra incorrecta o aleatoria.

La expectativa es que:

$$
s(x^+) > s(x^-)
$$

e idealmente que esa diferencia sea al menos un margen fijo.


###### **Pérdida hinge de ranking**

Para forzar esa separación, C\&W utiliza una pérdida de margen o **hinge loss**:

$$
\mathcal{L}_{\mathrm{C\&W}}
=
\sum_{x^+}\sum_{x^-}
\max\bigl(0,\,1 - s(x^+) + s(x^-)\bigr)
$$

Esta ecuación significa lo siguiente:

- si la ventana positiva ya tiene una puntuación al menos **1 unidad mayor** que la negativa, entonces no hay penalización,
- si esa diferencia no alcanza el margen, el modelo incurre en pérdida.


###### **Interpretación del término interno**

El término dentro del máximo es:

$$
1 - s(x^+) + s(x^-)
$$

y puede interpretarse así:

- $s(x^+)$ debería ser grande,
- $s(x^-)$ debería ser pequeña,
- la diferencia deseada es:

$$
s(x^+) - s(x^-) \ge 1
$$

Si esto se cumple, entonces:

$$
1 - s(x^+) + s(x^-) \le 0
$$

y la pérdida es cero:

$$
\max(0,\cdot)=0
$$

Si no se cumple, la pérdida es positiva y obliga al modelo a ajustar los parámetros para aumentar la puntuación de la secuencia correcta y/o disminuir la de la corrupta.

###### **Ejemplo intuitivo**

Supongamos que el modelo procesa:

- ventana real:
  $$
  x^+ = (\text{"el"}, \text{"gato"}, \text{"duerme"})
  $$
- ventana corrupta:
  $$
  x^- = (\text{"el"}, \text{"gato"}, \text{"mesa"})
  $$

Si el modelo asigna:

$$
s(x^+) = 2.3, \qquad s(x^-) = 0.8
$$

entonces:

$$
1 - s(x^+) + s(x^-)
=
1 - 2.3 + 0.8
=
-0.5
$$

y la pérdida es:

$$
\max(0,-0.5)=0
$$

No hay error porque la secuencia correcta ya supera a la incorrecta por más de 1 unidad.

En cambio, si:

$$
s(x^+) = 1.4, \qquad s(x^-) = 0.9
$$

entonces:

$$
1 - 1.4 + 0.9 = 0.5
$$

y la pérdida es:

$$
\max(0,0.5)=0.5
$$

Aquí el modelo sí es penalizado, porque la diferencia entre ambas puntuaciones todavía no alcanza el margen deseado.

###### **Cómo se entrena**

Los parámetros del modelo son:

$$
\theta = \{E, W, b_h, u, b_s\}
$$

donde:

- $E$ representa la matriz de embeddings,
- $W$ y $b_h$ definen la capa oculta,
- $u$ y $b_s$ producen la puntuación escalar.

Durante el entrenamiento, se minimiza la pérdida total sobre muchas ventanas positivas y negativas.  
Esto hace que:

- palabras que aparecen en contextos similares tiendan a tener embeddings similares,
- secuencias plausibles obtengan puntuaciones mayores que secuencias corruptas.


###### **Diferencia con NNLM**

La diferencia fundamental con el **NNLM** de Bengio es que C\&W:

- **no** calcula una distribución de probabilidad sobre todo el vocabulario,
- **no** usa softmax como salida principal,
- **sí** aprende una función escalar de compatibilidad o plausibilidad para una ventana.

Mientras el NNLM optimiza:

$$
P(w_t \mid \text{contexto})
$$

C\&W optimiza una restricción del tipo:

$$
s(x^+) > s(x^-)
$$

Esto lo hace computacionalmente más simple en algunos escenarios y conceptualmente muy importante como antecedente de muchos modelos posteriores de **ranking**, **matching** y **alineamiento multimodal**.


###### **Interpretación geométrica**

Puede interpretarse que el modelo aprende embeddings y parámetros de red tales que:

- ventanas lingüísticamente coherentes caen en regiones del espacio con alta puntuación,
- ventanas incoherentes caen en regiones con baja puntuación.

Así, el modelo no solo aprende vectores de palabras, sino una función que evalúa la **compatibilidad contextual** de una secuencia.


###### **Importancia para modelos posteriores**

La idea de C\&W fue influyente porque introdujo con claridad el uso de:

- embeddings aprendidos conjuntamente,
- ventanas locales de contexto,
- una función de puntuación escalar,
- y una pérdida de ranking con margen.

Este esquema anticipa muchas formulaciones posteriores en NLP y multimodalidad, donde el objetivo ya no es únicamente predecir palabras, sino aprender que un par o una estructura correcta debe puntuar mejor que una incorrecta.

In [ ]:
import numpy as np

vocab = ["I", "love", "cats", "dogs", "and"]
word2idx = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
d = 20
window_size = 3

np.random.seed(42)
E = np.random.randn(V, d)
w_score = np.random.randn(window_size * d)

def score(window):
    concat_emb = np.hstack([E[word2idx[w]] for w in window])
    return float(concat_emb.dot(w_score))

x_pos = ["I", "love", "cats"]
x_neg = ["I", "dogs", "cats"]

s_pos = score(x_pos)
s_neg = score(x_neg)
loss = max(0.0, 1 - s_pos + s_neg)

print("score(x+) =", round(s_pos, 4))
print("score(x-) =", round(s_neg, 4))
print("hinge loss =", round(loss, 4))

#### **3.3 CBOW y Skip-gram**

**Word2Vec** popularizó dos tareas auto-supervisadas para aprender embeddings de palabras:

- **CBOW (Continuous Bag of Words)**: usa las palabras del contexto para predecir la palabra central.
- **Skip-gram**: usa la palabra central para predecir las palabras del contexto.

Aunque ambas tareas son distintas, comparten la misma idea central:  
las palabras que aparecen en contextos parecidos deben terminar con representaciones vectoriales parecidas.

##### **Idea general**

Supongamos una secuencia de palabras:

$$
(w_{t-c}, \dots, w_{t-1}, w_t, w_{t+1}, \dots, w_{t+c})
$$

donde:

- $w_t$ es la palabra central,
- $c$ es el radio de la ventana de contexto.

Por ejemplo, si la oración es:

> *el gato negro duerme en la silla*

y tomamos como palabra central:

$$
w_t = \text{"duerme"}
$$

con ventana $c=2$, entonces el contexto podría ser:

$$
\{\text{"gato"}, \text{"negro"}, \text{"en"}, \text{"la"}\}
$$


#### **CBOW**

En **CBOW**, el modelo toma las palabras del contexto y trata de predecir la palabra central.

Formalmente:

$$
(w_{t-c}, \dots, w_{t-1}, w_{t+1}, \dots, w_{t+c})
\longrightarrow w_t
$$

##### **Paso 1. Embeddings del contexto**

Cada palabra del contexto tiene un embedding de entrada:

$$
v_{w_{t-c}}, \dots, v_{w_{t-1}}, v_{w_{t+1}}, \dots, v_{w_{t+c}}
$$

donde cada vector pertenece a $\mathbb{R}^d$.

##### **Paso 2. Agregación del contexto**

En CBOW, los embeddings del contexto suelen agregarse mediante suma o promedio:

$$
v_C = \frac{1}{2c}\sum_{\substack{-c \le j \le c \\ j \ne 0}} v_{w_{t+j}}
$$

Aquí:

- $v_C$ es la representación agregada del contexto,
- si se usa suma en vez de promedio, la idea matemática es la misma.

##### **Paso 3. Predicción de la palabra central**

Con una salida softmax completa, el modelo estimaría:

$$
P(w_t \mid \text{contexto})
=
\frac{\exp(u_{w_t}^{\top} v_C)}
{\sum_{w \in V} \exp(u_w^{\top} v_C)}
$$

donde:

- $u_w$ es el embedding de salida de la palabra $w$,
- $V$ es el vocabulario.

Sin embargo, calcular esa suma sobre todo el vocabulario es costoso.  
Por eso Word2Vec usa con frecuencia **negative sampling**.


#### **Skip-gram**

En **Skip-gram**, el modelo hace lo contrario: toma la palabra central y trata de predecir las palabras del contexto.

Formalmente:

$$
w_t \longrightarrow (w_{t-c}, \dots, w_{t-1}, w_{t+1}, \dots, w_{t+c})
$$

##### **Paso 1. Embedding de la palabra central**

La palabra central tiene un embedding de entrada:

$$
v_{w_t}
$$

##### **Paso 2. Predicción de cada palabra de contexto**

Para cada palabra de contexto $w_{t+j}$, el modelo quiere que la palabra real tenga una puntuación alta.

Con softmax completa:

$$
P(w_{t+j} \mid w_t)
=
\frac{\exp(u_{w_{t+j}}^{\top} v_{w_t})}
{\sum_{w \in V} \exp(u_w^{\top} v_{w_t})}
$$

Otra vez, esta formulación es costosa si el vocabulario es grande, por lo que se suele reemplazar por **negative sampling**.

#### **Embeddings de entrada y de salida**

Word2Vec aprende dos conjuntos de vectores:

- **embeddings de entrada**: $v_w$
- **embeddings de salida**: $u_w$

Esto significa que cada palabra tiene dos representaciones durante el entrenamiento:

- una cuando aparece como entrada,
- otra cuando aparece como objetivo.

Al final, muchas implementaciones usan solo $v_w$, o combinan ambos.

#### **Negative Sampling**

La idea de **negative sampling** es convertir el problema en una tarea de clasificación binaria local:

- el par correcto (**positivo**) debe puntuar alto,
- varios pares incorrectos (**negativos**) deben puntuar bajo.

La puntuación entre dos palabras o entre una palabra y un contexto se calcula con un producto interno:

$$
u_w^\top v_C
$$

o, en Skip-gram:

$$
u_w^\top v_{w_t}
$$

Si este valor es alto, el modelo considera que el par es compatible.

##### **Función sigmoide**

La función usada para convertir esa puntuación en algo interpretable como probabilidad es la sigmoide:

$$
\sigma(x)=\frac{1}{1+e^{-x}}
$$

Propiedades importantes:

- si $x$ es grande y positivo, $\sigma(x)\approx 1$,
- si $x$ es grande y negativo, $\sigma(x)\approx 0$.

Por tanto:

- queremos $\sigma(u_{w_t}^\top v_C)$ cerca de 1 para el par positivo,
- queremos $\sigma(u_{w_i}^\top v_C)$ cerca de 0 para los negativos.


##### **Objetivo con Negative Sampling**

Una forma estándar y simple del objetivo es:

$$
\mathcal{L}_{\text{NS}}
=
-\log \sigma\!\big(u_{w_t}^{\top} v_C\big)
-
\sum_{i=1}^{k}\log \sigma\!\big(-u_{w_i}^{\top} v_C\big),
$$

donde:

- $v_C$ es el embedding del contexto agregado (**CBOW**) o de la palabra central (**Skip-gram**),
- $u_{w_t}$ es el embedding de salida de la palabra positiva,
- $w_i$ son muestras negativas,
- $k$ es el número de negativos.

#### **Interpretación término por término**

##### **Primer término: ejemplo positivo**

$$
-\log \sigma\!\big(u_{w_t}^{\top} v_C\big)
$$

Este término penaliza al modelo cuando el par correcto no recibe suficiente compatibilidad.

Si el producto interno:

$$
u_{w_t}^{\top} v_C
$$

es grande, entonces:

$$
\sigma(u_{w_t}^{\top} v_C)\approx 1
$$

y la pérdida es pequeña.

##### **Segundo término: ejemplos negativos**

$$
-\sum_{i=1}^{k}\log \sigma\!\big(-u_{w_i}^{\top} v_C\big)
$$

Aquí queremos que cada palabra negativa tenga puntuación baja frente al contexto.

Si $u_{w_i}^{\top} v_C$ es muy negativa, entonces:

$$
\sigma(-u_{w_i}^{\top} v_C)\approx 1
$$

y la pérdida también será pequeña.

En otras palabras:

- el par positivo debe acercarse,
- los pares negativos deben alejarse.

##### **Ejemplo intuitivo en CBOW**

Supongamos que el contexto es:

$$
\{\text{"el"}, \text{"gato"}, \text{"negro"}, \text{"en"}\}
$$

y la palabra central correcta es:

$$
w_t = \text{"duerme"}
$$

Entonces:

- el modelo construye $v_C$ agregando los embeddings del contexto,
- toma como positivo el par $(v_C, u_{\text{duerme}})$,
- y como negativos palabras aleatorias, por ejemplo:
  - "árbol"
  - "teléfono"
  - "nube"

La pérdida obliga a que:

$$
u_{\text{duerme}}^\top v_C
$$

sea mayor que:

$$
u_{\text{árbol}}^\top v_C,\quad
u_{\text{teléfono}}^\top v_C,\quad
u_{\text{nube}}^\top v_C
$$

##### **Ejemplo numérico simple**

Supongamos:

- positivo:
  $$
  u_{w_t}^{\top} v_C = 2.0
  $$
- negativos:
  $$
  u_{w_1}^{\top} v_C = -1.5,\qquad
  u_{w_2}^{\top} v_C = -0.8
  $$

Entonces:

$$
\sigma(2.0) \approx 0.881
$$

y:

$$
-\log(0.881)\approx 0.127
$$

Para el primer negativo:

$$
\sigma(-(-1.5))=\sigma(1.5)\approx 0.818
$$

$$
-\log(0.818)\approx 0.201
$$

Para el segundo negativo:

$$
\sigma(-(-0.8))=\sigma(0.8)\approx 0.690
$$

$$
-\log(0.690)\approx 0.371
$$

La pérdida total sería aproximadamente:

$$
0.127 + 0.201 + 0.371 = 0.699
$$

Si los negativos estuvieran todavía más lejos y el positivo más cerca, la pérdida sería menor.

#### **Cómo cambia entre CBOW y Skip-gram**

La fórmula puede verse igual, pero cambia la interpretación de $v_C$.

##### **En CBOW**
$ v_C $ representa el **contexto agregado**:

$$
v_C = \frac{1}{2c}\sum_{\substack{-c \le j \le c \\ j \ne 0}} v_{w_{t+j}}
$$

y se usa para predecir la palabra central $w_t$.

##### **En Skip-gram**
$ v_C $ puede interpretarse simplemente como el embedding de la palabra central:

$$
v_C = v_{w_t}
$$

y la pérdida se aplica una vez por cada palabra de contexto positiva.

Es decir, en Skip-gram entrenamos pares del tipo:

$$
(w_t, w_{t+j})
$$

para cada contexto $j$ dentro de la ventana.

#### **Diferencia conceptual entre CBOW y Skip-gram**

**CBOW**
- agrega la información del contexto,
- suele ser más rápido,
- funciona bien con palabras frecuentes.

**Skip-gram**
- aprende a partir de pares palabra-contexto,
- suele capturar mejor palabras raras,
- genera más ejemplos de entrenamiento.

#### **Relación con la matriz PMI**

Una interpretación importante es que Word2Vec, especialmente Skip-gram con negative sampling, está relacionado con la factorización implícita de una matriz de coocurrencias o de tipo **PMI desplazado**.  
Eso ayuda a entender por qué estos modelos aprenden relaciones semánticas útiles: están explotando patrones de coocurrencia entre palabras y contextos.

##### **Importancia para modelos posteriores**

CBOW y Skip-gram fueron fundamentales porque mostraron que era posible aprender embeddings de calidad con objetivos simples y eficientes.  
Además, muchas ideas posteriores en NLP y multimodalidad retoman esta lógica:

- pares positivos y negativos,
- compatibilidad por producto interno,
- aprendizaje de espacios vectoriales compartidos.

Por eso también son un antecedente importante para modelos de alineamiento multimodal y aprendizaje contrastivo.

In [ ]:
import numpy as np

vocab = ["I", "like", "NLP", "and", "AI"]
word2idx = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
d = 5

np.random.seed(0)
W_in = np.random.randn(V, d)
W_out = np.random.randn(V, d)

freqs = np.array([1, 2, 1, 1, 1], dtype=np.float32)
p_noise = freqs**0.75
p_noise /= p_noise.sum()

sigma = lambda x: 1 / (1 + np.exp(-x))

context = ["I", "NLP", "and", "AI"]
target = "like"

v_C = np.mean([W_in[word2idx[w]] for w in context], axis=0)
u_t = W_out[word2idx[target]]
pos_term = -np.log(sigma(u_t.dot(v_C)))

k = 2
neg_idxs = np.random.choice(V, size=k, p=p_noise)
neg_term = -np.sum(np.log(sigma(-W_out[neg_idxs].dot(v_C))))

loss = pos_term + neg_term
print("Negative sampling loss =", round(float(loss), 4))

##### **3.4 Subpalabras y morfología**

Uno de los problemas clásicos en los modelos de embeddings de palabras es que el vocabulario puede ser:

- muy grande,
- morfológicamente rico,
- y además contener muchas palabras raras o no vistas (**OOV**, *out of vocabulary*).

Por ejemplo, en español, palabras como:

- *gato*,
- *gatos*,
- *gatito*,
- *gatitos*,
- *gatuno*

comparten una raíz y parte de su estructura morfológica, pero un modelo que represente cada palabra como una unidad completamente independiente tendría que aprender un vector distinto para cada una, sin explotar explícitamente esas relaciones internas.

Para manejar este problema, una estrategia estándar consiste en representar cada palabra no solo como una unidad completa, sino también a partir de sus **subunidades**, por ejemplo **n-gramas de caracteres**.

Una formulación inspirada en **FastText** es:

$$
v_w = \frac{1}{|G(w)|+1}\left(z_w + \sum_{g\in G(w)} z_g\right),
$$

donde:

- $z_w$ es el embedding de la palabra completa,
- $G(w)$ es el conjunto de n-gramas de caracteres de $w$,
- $z_g$ es el embedding del n-grama $g$,
- $v_w$ es la representación final de la palabra.

##### **Idea general**

En lugar de aprender un vector únicamente para la palabra completa, el modelo asume que una palabra puede descomponerse en piezas menores que contienen información útil.

Así, la representación de una palabra combina:

1. información global de la palabra completa,
2. información subléxica o morfológica de sus fragmentos.

Esto permite que palabras parecidas en forma compartan parte de su representación, incluso si una de ellas aparece pocas veces o no apareció en entrenamiento.

###### **¿Qué significa $G(w)$?**

El conjunto $G(w)$ contiene los **n-gramas de caracteres** extraídos de la palabra $w$.

Un n-grama de caracteres es simplemente una subsecuencia contigua de longitud fija.

Por ejemplo, si:

$$
w = \text{"gatos"}
$$

y usamos trigramas de caracteres, podríamos extraer:

$$
G(w)=\{\text{"gat"}, \text{"ato"}, \text{"tos"}\}
$$

En FastText suele ser común trabajar con varios tamaños de n-gramas, por ejemplo de 3 a 6 caracteres, e incluso añadir símbolos de borde para distinguir prefijos y sufijos.

Por ejemplo, si escribimos:

$$
\langle gatos \rangle
$$

podrían aparecer n-gramas como:

- $\langle ga$
- gat
- ato
- tos
- os \rangle

Esto ayuda a capturar información morfológica como:

- prefijos,
- sufijos,
- raíces,
- variaciones flexivas.

##### **Interpretación de la ecuación**

La representación final de la palabra es:

$$
v_w = \frac{1}{|G(w)|+1}\left(z_w + \sum_{g\in G(w)} z_g\right)
$$

Esta ecuación puede leerse así:

- se toma el embedding de la palabra completa, $z_w$,
- se suman los embeddings de todos sus n-gramas, $z_g$,
- luego se normaliza dividiendo por el número total de componentes, es decir, $|G(w)|+1$.

La división por $|G(w)|+1$ convierte la suma en una especie de **promedio**, de modo que la magnitud del vector no crezca simplemente porque una palabra tenga más subunidades.

##### **Desglose paso a paso**

Supongamos que una palabra $w$ tiene tres n-gramas:

$$
G(w)=\{g_1,g_2,g_3\}
$$

Entonces su vector sería:

$$
v_w=\frac{1}{4}(z_w + z_{g_1}+z_{g_2}+z_{g_3})
$$

Aquí aparecen cuatro términos en total:

- el embedding de la palabra completa,
- y tres embeddings de subunidades.

Por eso el denominador es 4.

##### **Ejemplo numérico simple**

Supongamos que queremos representar la palabra:

$$
w=\text{"gatos"}
$$

y que sus trigramas son:

$$
G(w)=\{\text{"gat"}, \text{"ato"}, \text{"tos"}\}
$$

Supongamos además que los embeddings tienen dimensión 2 y valen:

$$
z_w=
\begin{bmatrix}
0.8\\
0.4
\end{bmatrix},
\qquad
z_{\text{gat}}=
\begin{bmatrix}
0.7\\
0.1
\end{bmatrix},
\qquad
z_{\text{ato}}=
\begin{bmatrix}
0.5\\
0.3
\end{bmatrix},
\qquad
z_{\text{tos}}=
\begin{bmatrix}
0.2\\
0.6
\end{bmatrix}
$$

Entonces:

$$
z_w + z_{\text{gat}} + z_{\text{ato}} + z_{\text{tos}}
=
\begin{bmatrix}
0.8\\0.4
\end{bmatrix}
+
\begin{bmatrix}
0.7\\0.1
\end{bmatrix}
+
\begin{bmatrix}
0.5\\0.3
\end{bmatrix}
+
\begin{bmatrix}
0.2\\0.6
\end{bmatrix}
=
\begin{bmatrix}
2.2\\1.4
\end{bmatrix}
$$

Como hay 3 n-gramas más la palabra completa, dividimos entre 4:

$$
v_w=
\frac{1}{4}
\begin{bmatrix}
2.2\\1.4
\end{bmatrix}
=
\begin{bmatrix}
0.55\\0.35
\end{bmatrix}
$$

Ese sería el embedding final de la palabra.

##### **¿Por qué esto ayuda con la morfología?**

Porque muchas palabras comparten fragmentos internos.

Por ejemplo:

- *cantar*
- *cantando*
- *cantante*
- *cantamos*

comparten subunidades como:

- *cant*
- *anta*
- *ando*
- *amos*

Si el modelo aprende buenos embeddings para esos fragmentos, entonces palabras relacionadas morfológicamente tenderán a compartir parte de su representación.

Esto es especialmente útil en lenguas con:

- mucha flexión verbal,
- variación de género y número,
- derivación rica,
- composición de palabras.

##### **¿Por qué esto ayuda con palabras OOV?**

Supongamos que una palabra no apareció en entrenamiento, por ejemplo:

$$
w = \text{"hiperconectividad"}
$$

Si el modelo no tiene un embedding aprendido para la palabra completa, aún puede aproximar su representación a partir de subunidades conocidas, por ejemplo:

- *hiper*
- *conect*
- *ividad*

o, más formalmente, a partir de n-gramas de caracteres que sí estuvieron presentes durante el entrenamiento.

En ese caso, aunque $z_w$ no esté disponible o sea poco fiable, la suma de los $z_g$ ya proporciona una representación útil.

Esta es una de las grandes ventajas frente a modelos de palabra completa como Word2Vec clásico.

##### **Relación con FastText**

FastText generaliza la idea de Word2Vec incorporando subpalabras.

En Word2Vec clásico, cada palabra tiene un embedding independiente:

$$
w \mapsto v_w
$$

En FastText, la palabra se representa como suma o promedio de:

- su embedding global,
- y los embeddings de sus n-gramas.

Así, dos palabras distintas pueden compartir parte de su estructura vectorial si comparten fragmentos de caracteres.

##### **Interpretación geométrica**

La ecuación puede verse como una forma de construir el vector de una palabra mediante una **composición aditiva** de señales:

- una señal léxica global: $z_w$,
- varias señales subléxicas: $z_g$.

En el espacio vectorial, esto implica que palabras con morfología parecida quedan más cerca entre sí, porque comparten varios términos de la suma.

Por eso, por ejemplo, *gato* y *gatos* tenderán a estar más cerca que *gato* y *avión*.

##### **Caso sin embedding de palabra completa**

En algunas variantes, puede usarse solo la suma de subpalabras:

$$
v_w = \frac{1}{|G(w)|}\sum_{g\in G(w)} z_g
$$

Esto es útil especialmente para palabras raras u OOV.

Sin embargo, la versión con $z_w$ incluido suele ser más estable cuando la palabra completa sí aparece con frecuencia suficiente en entrenamiento.

##### **Relación con CBOW y Skip-gram**

La fórmula anterior define **cómo se representa la palabra**.  
Luego, ese vector $v_w$ puede insertarse dentro de objetivos como:

- **CBOW**, donde se agregan embeddings del contexto para predecir una palabra
- **Skip-gram**, donde una palabra predice palabras de contexto.

La diferencia es que ahora el embedding de entrada de una palabra ya no es un vector atómico aprendido de forma aislada, sino una composición de subunidades.

Así, en vez de usar directamente:

$$
v_w
$$

como una fila independiente de una matriz de embeddings, se usa una representación construida a partir de:

$$
z_w \quad \text{y} \quad \{z_g : g\in G(w)\}
$$

##### **Importancia para NLP y multimodalidad**

El uso de subpalabras fue crucial porque permitió:

- manejar vocabularios grandes,
- reducir el problema de palabras fuera de vocabulario,
- capturar regularidades morfológicas,
- mejorar el aprendizaje en lenguas morfológicamente.

Además, esta idea conecta con enfoques posteriores de tokenización subléxica y segmentación usados en modelos modernos, aunque allí ya no se trabaja exactamente con n-gramas de caracteres, sino con unidades aprendidas de manera más flexible.

En términos conceptuales, la idea de subpalabras muestra que una representación útil no siempre debe estar asociada a una palabra entera: también puede construirse componiendo unidades menores.

In [ ]:
import numpy as np

def extract_ngrams(word, min_n=3, max_n=5):
    word = f"<{word}>"
    grams = []
    for n in range(min_n, max_n + 1):
        for i in range(len(word) - n + 1):
            grams.append(word[i:i+n])
    return grams

word = "running"
ngrams = extract_ngrams(word)

np.random.seed(1)
emb_dim = 8
z_w = np.random.randn(emb_dim)
z_g = {g: np.random.randn(emb_dim) for g in ngrams}

v_w = (z_w + sum(z_g[g] for g in ngrams)) / (len(ngrams) + 1)
print("Número de n-gramas:", len(ngrams))
print("Embedding híbrido:", np.round(v_w, 3))

#### **4. Representaciones de frases**

Cuando pasamos de palabras aisladas a frases, necesitamos una forma de construir una representación única para toda la secuencia. 

La idea general es tomar los embeddings de las palabras:

$$
S=(w_1,\dots,w_n), \qquad v_i = e_{w_i}\in\mathbb{R}^d
$$

y componerlos en un solo vector:

$$
v_S \in \mathbb{R}^d
$$

que resuma el contenido semántico de la frase.

Las estrategias más simples y clásicas son:

- **promedio simple** de embeddings,
- **promedio ponderado**,
- y variantes como **SIF (Smooth Inverse Frequency)**.

##### **4.1 Promedio simple**

Sea una frase:

$$
S=(w_1,\dots,w_n)
$$

con embeddings:

$$
v_i=e_{w_i}
$$

Una representación muy simple de la frase es el promedio de sus vectores:

$$
v_S = \frac{1}{n}\sum_{i=1}^{n} v_i
$$

donde:

- $n$ es el número de palabras de la frase,
- $v_i \in \mathbb{R}^d$ es el embedding de la palabra $w_i$,
- $v_S \in \mathbb{R}^d$ es el embedding resultante de la frase.


##### **¿Qué significa esta ecuación?**

La ecuación dice que la frase se representa como el **centroide** o promedio de los embeddings de sus palabras.

Es decir:

1. se toman todos los vectores de palabras,
2. se suman componente a componente,
3. se divide entre el número total de palabras.

La intuición es que si los embeddings de las palabras ya codifican significado semántico, entonces su promedio puede dar una señal razonable del contenido global de la frase.

##### **Ejemplo numérico**

Supongamos que una frase tiene tres palabras:

$$
S=(w_1,w_2,w_3)
$$

con embeddings de dimensión 2:

$$
v_1=
\begin{bmatrix}
0.8\\
0.2
\end{bmatrix},
\qquad
v_2=
\begin{bmatrix}
0.4\\
0.6
\end{bmatrix},
\qquad
v_3=
\begin{bmatrix}
0.1\\
0.5
\end{bmatrix}
$$

Entonces:

$$
v_S = \frac{1}{3}(v_1+v_2+v_3)
$$

Primero sumamos:

$$
v_1+v_2+v_3=
\begin{bmatrix}
0.8\\0.2
\end{bmatrix}
+
\begin{bmatrix}
0.4\\0.6
\end{bmatrix}
+
\begin{bmatrix}
0.1\\0.5
\end{bmatrix}
=
\begin{bmatrix}
1.3\\1.3
\end{bmatrix}
$$

Ahora dividimos entre 3:

$$
v_S=
\frac{1}{3}
\begin{bmatrix}
1.3\\1.3
\end{bmatrix}
=
\begin{bmatrix}
0.433\\0.433
\end{bmatrix}
$$

Ese sería el embedding de la frase.

###### **Ventajas del promedio simple**

- Es muy barato computacionalmente,
- No requiere entrenamiento adicional,
- Funciona sorprendentemente bien como línea base,
- Es útil en tareas de similitud semántica, clustering o recuperación.

##### **Limitación principal**

El promedio simple **ignora el orden de las palabras**.

Por ejemplo, las frases:

- *"perro muerde hombre"*
- *"hombre muerde perro"*

podrían quedar muy parecidas si contienen las mismas palabras, aunque su significado sea claramente distinto.

También da el mismo peso a palabras muy informativas y a palabras muy frecuentes o poco útiles, como artículos o preposiciones.

##### **4.2 Promedio ponderado y SIF**

Para mejorar el promedio simple, una idea natural es no tratar todas las palabras por igual.

Las palabras muy frecuentes, como:

- *el*,
- *la*,
- *de*,
- *y*,

aparecen en muchísimas frases y suelen aportar menos información semántica específica que palabras de contenido como:

- *gato*,
- *neural*,
- *transformer*,
- *multimodal*.

Por eso, una variante ampliamente usada consiste en ponderar cada palabra según su frecuencia en el corpus.

Una forma clásica es:

$$
\alpha_i = \frac{a}{a+p(w_i)},
\qquad
v_S = \frac{1}{n}\sum_{i=1}^{n}\alpha_i v_i
$$

donde:

- $p(w_i)$ es la frecuencia o probabilidad estimada de la palabra $w_i$ en el corpus,
- $a$ es una constante pequeña positiva, por ejemplo $a=10^{-3}$,
- $\alpha_i$ es el peso asignado a la palabra $w_i$.

##### **Interpretación del peso $\alpha_i$**

La fórmula:

$$
\alpha_i = \frac{a}{a+p(w_i)}
$$

tiene un comportamiento muy útil:

- si $p(w_i)$ es grande, entonces $\alpha_i$ es pequeño,
- si $p(w_i)$ es pequeño, entonces $\alpha_i$ es mayor.

Es decir:

- palabras muy frecuentes reciben menos peso,
- palabras menos frecuentes y más informativas reciben más peso.

##### **Ejemplo intuitivo**

Supongamos dos palabras:

- *el* con frecuencia alta:
  $$
  p(\text{el}) = 0.05
  $$
- *transformer* con frecuencia baja:
  $$
  p(\text{transformer}) = 0.0001
  $$

si tomamos:

$$
a=0.001
$$

entonces:

Para *el*:

$$
\alpha_{\text{el}}=
\frac{0.001}{0.001+0.05}
\approx 0.0196
$$

Para *transformer*:

$$
\alpha_{\text{transformer}}=
\frac{0.001}{0.001+0.0001}
\approx 0.909
$$

Así, la palabra frecuente *el* casi no pesa, mientras que *transformer* tiene mucho mayor influencia en la representación final.

##### **Promedio ponderado**

Una vez obtenidos los pesos, el embedding de la frase queda:

$$
v_S = \frac{1}{n}\sum_{i=1}^{n}\alpha_i v_i
$$

Esto significa que no todas las palabras contribuyen por igual al promedio.

Palabras funcionales y muy frecuentes se atenúan, mientras que palabras de contenido tienen un papel más fuerte.

##### **Ejemplo numérico simple**

Supongamos una frase con dos palabras y embeddings de dimensión 2:

$$
v_1=
\begin{bmatrix}
1.0\\
0.5
\end{bmatrix},
\qquad
v_2=
\begin{bmatrix}
0.2\\
0.8
\end{bmatrix}
$$

y pesos:

$$
\alpha_1 = 0.1,
\qquad
\alpha_2 = 0.9
$$

Entonces:

$$
v_S = \frac{1}{2}\left(\alpha_1 v_1 + \alpha_2 v_2\right)
$$

Calculamos:

$$
\alpha_1 v_1=
0.1
\begin{bmatrix}
1.0\\0.5
\end{bmatrix}
=
\begin{bmatrix}
0.1\\0.05
\end{bmatrix}
$$

$$
\alpha_2 v_2=
0.9
\begin{bmatrix}
0.2\\0.8
\end{bmatrix}
=
\begin{bmatrix}
0.18\\0.72
\end{bmatrix}
$$

Sumamos:

$$
\begin{bmatrix}
0.1\\0.05
\end{bmatrix}
+
\begin{bmatrix}
0.18\\0.72
\end{bmatrix}
=
\begin{bmatrix}
0.28\\0.77
\end{bmatrix}
$$

Dividimos entre 2:

$$
v_S=
\begin{bmatrix}
0.14\\0.385
\end{bmatrix}
$$

Se ve claramente que la segunda palabra domina más la representación final.

##### **SIF: Smooth Inverse Frequency**

El método **SIF** (*Smooth Inverse Frequency*) va un paso más allá del promedio ponderado.  
Después de construir el vector de la frase, elimina una dirección dominante común en el espacio vectorial.

La idea es que muchas frases comparten una componente global asociada a palabras muy frecuentes o patrones generales del lenguaje.  
Esa componente común puede hacer que muchas frases queden artificialmente cerca entre sí.

Por eso, una vez obtenido $v_S$, se calcula:

$$
v'_S = v_S - u(u^\top v_S)
$$

donde:

- $u \in \mathbb{R}^d$ es la primera componente principal estimada sobre muchos embeddings de frases,
- $u^\top v_S$ es la proyección escalar de la frase sobre esa dirección,
- $u(u^\top v_S)$ es la proyección vectorial de la frase sobre $u$,
- $v'_S$ es el embedding corregido de la frase.

##### **Interpretación geométrica de SIF**

La operación:

$$
u(u^\top v_S)
$$

extrae la parte de $v_S$ que está alineada con la dirección principal $u$.

Entonces:

$$
v'_S = v_S - u(u^\top v_S)
$$

significa que restamos esa componente compartida.

Geométricamente:

- $v_S$ es el vector original de la frase,
- $u$ es una dirección global muy dominante,
- $v'_S$ es el vector "descontaminado" de esa señal común.

Esto suele mejorar la calidad semántica de la representación porque reduce la influencia de información demasiado general.

##### **Ejemplo geométrico simple**

Supongamos que una frase tiene representación:

$$
v_S=
\begin{bmatrix}
2\\1
\end{bmatrix}
$$

y que la primera componente principal es:

$$
u=
\begin{bmatrix}
1\\0
\end{bmatrix}
$$

Entonces:

$$
u^\top v_S =
\begin{bmatrix}
1 & 0
\end{bmatrix}
\begin{bmatrix}
2\\1
\end{bmatrix}
=2
$$

y la proyección sobre $u$ es:

$$
u(u^\top v_S)=
\begin{bmatrix}
1\\0
\end{bmatrix}
(2)
=
\begin{bmatrix}
2\\0
\end{bmatrix}
$$

Por tanto:

$$
v'_S = v_S - u(u^\top v_S)
=
\begin{bmatrix}
2\\1
\end{bmatrix}
-
\begin{bmatrix}
2\\0
\end{bmatrix}
=
\begin{bmatrix}
0\\1
\end{bmatrix}
$$

La componente asociada a la dirección dominante fue eliminada.

##### **Por qué funciona SIF**

En muchos espacios de embeddings, la primera componente principal está muy asociada a:

- palabras frecuentes,
- estructura general del idioma,
- información poco discriminativa.

Al eliminarla, las frases suelen separarse mejor semánticamente.

Por eso SIF ha sido una técnica muy popular como línea base fuerte para representaciones de frases.

##### **Comparación entre los métodos**

**Promedio simple**
- muy barato,
- fácil de implementar,
- ignora frecuencia y orden.

**Promedio ponderado**
- sigue siendo simple,
- reduce el peso de palabras demasiado frecuentes,
- mejora la señal semántica.

**SIF**
- añade una corrección geométrica posterior,
- elimina una dirección dominante común,
- suele producir mejores embeddings de frases sin usar modelos complejos.

##### **Limitaciones**

Aunque estos métodos funcionan bien como líneas base, siguen teniendo limitaciones importantes:

- ignoran casi por completo el orden de las palabras,
- no modelan composición sintáctica compleja,
- no capturan bien negación, dependencia larga o estructura gramatical,
- dependen de embeddings de palabras ya aprendidos.

Por eso, posteriormente se desarrollaron modelos más sofisticados para frases y oraciones, como:

- RNNs,
- CNNs para texto,
- BiLSTM con pooling,
- transformers,
- embeddings contextuales.

##### **Importancia conceptual**

Aun así, estas representaciones de frases son muy importantes porque muestran una idea fundamental:

> una representación útil de una secuencia puede construirse componiendo representaciones de unidades más pequeñas.

Ese principio aparece una y otra vez en NLP y también en multimodalidad, donde luego se busca componer:

- palabras en frases,
- frases en documentos,
- o incluso texto e imagen en un espacio compartido.

In [ ]:
import numpy as np

words = ["deep", "learning", "models", "learn"]
np.random.seed(0)
embs = {w: np.random.randn(4) for w in words}
freq = {"deep": 0.001, "learning": 0.01, "models": 0.02, "learn": 0.005}

a = 1e-3
alphas = {w: a / (a + freq[w]) for w in words}
v_S = sum(alphas[w] * embs[w] for w in words) / len(words)

print("Pesos SIF:", {w: round(alphas[w], 4) for w in words})
print("Vector de frase:", np.round(v_S, 3))

#### **5. Representaciones de oraciones**

A diferencia de las representaciones de frases por promedio, a nivel de **oración** ya interesa capturar:

- **orden de las palabras**,
- **dependencias de largo alcance**,
- **composición sintáctica**,
- y una semántica más rica.

Por eso se usan codificadores secuenciales como **LSTM**, **BiLSTM** y luego mecanismos de **pooling** o **atención** para obtener un solo vector de toda la oración.

##### **5.1 BiLSTM + max-pooling (estilo InferSent)**

La idea de un codificador **bidireccional** es procesar la oración en dos direcciones:

- de izquierda a derecha,
- de derecha a izquierda.

Así, cada posición puede incorporar información del contexto pasado y futuro.

Sea una oración:

$$
S=(w_1,w_2,\dots,w_n)
$$

y sea $e_{w_t}\in\mathbb{R}^d$ el embedding de la palabra $w_t$.

##### **Paso 1. LSTM hacia adelante**

La LSTM hacia adelante recorre la oración desde el inicio hasta el final:

$$
\overrightarrow{h}_t = \mathrm{LSTM}_f(\overrightarrow{h}_{t-1}, e_{w_t})
$$

Aquí:

- $\overrightarrow{h}_t$ es el estado oculto en la posición $t$ al leer de izquierda a derecha,
- $\overrightarrow{h}_{t-1}$ resume la información acumulada hasta la palabra anterior,
- $e_{w_t}$ es el embedding de la palabra actual.

La intuición es que $\overrightarrow{h}_t$ resume el prefijo:

$$
(w_1,\dots,w_t)
$$

##### **Paso 2. LSTM hacia atrás**

La LSTM hacia atrás recorre la oración desde el final hacia el inicio:

$$
\overleftarrow{h}_t = \mathrm{LSTM}_b(\overleftarrow{h}_{t+1}, e_{w_t})
$$

Aquí:

- $\overleftarrow{h}_t$ es el estado oculto en la posición $t$ al leer de derecha a izquierda,
- $\overleftarrow{h}_{t+1}$ resume la información del sufijo ya leído desde la derecha.

La intuición es que $\overleftarrow{h}_t$ resume:

$$
(w_t,\dots,w_n)
$$

##### **¿Por qué usar una BiLSTM?**

Porque una palabra dentro de una oración normalmente depende tanto del contexto anterior como del posterior.

Por ejemplo, en la palabra:

> *banco*

su interpretación puede depender de palabras que vienen antes y también después:

- *banco central*
- *banco del río*

Una representación solo hacia adelante puede perder parte de esa información futura.  
La representación bidireccional la recupera.

##### **Paso 3. Concatenación de ambas direcciones**

En cada posición $t$, se combinan los dos estados ocultos:

$$
h_t = [\overrightarrow{h}_t, \overleftarrow{h}_t]
$$

donde $[\,,\,]$ indica concatenación.

Si:

- $\overrightarrow{h}_t\in\mathbb{R}^m$
- $\overleftarrow{h}_t\in\mathbb{R}^m$

entonces:

$$
h_t \in \mathbb{R}^{2m}
$$

Esta representación contiene, para la posición $t$:

- contexto izquierdo,
- contexto derecho,
- y la palabra actual integrada dentro de ambos.

###### **Interpretación de $h_t$**

Cada vector $h_t$ ya no representa solo la palabra $w_t$, sino la palabra **en contexto**.

Esto es importante: el modelo deja de tener una representación aislada por palabra y pasa a producir una representación contextualizada por posición.

Así, la secuencia:

$$
h_1,h_2,\dots,h_n
$$

es una representación contextual de toda la oración.

##### **Paso 4. Max-pooling por dimensión**

Para obtener un único vector de toda la oración, una estrategia muy usada es aplicar **max-pooling** sobre la secuencia de estados:

$$
v_{\text{sent}}(j) = \max_{1\le t\le n} h_t(j)
$$

donde:

- $h_t(j)$ es la componente $j$-ésima del vector $h_t$,
- $v_{\text{sent}}(j)$ es la componente $j$-ésima del embedding final de la oración.

En palabras: para cada dimensión $j$, se toma el mayor valor observado en toda la oración.

###### **¿Qué hace realmente el max-pooling?**

Si cada $h_t$ tiene dimensión $2m$, entonces la oración produce una matriz:

$$
H=
\begin{bmatrix}
h_1^\top\\
h_2^\top\\
\vdots\\
h_n^\top
\end{bmatrix}
\in \mathbb{R}^{n\times 2m}
$$

El max-pooling toma el máximo por columna.  
Así, el vector final de la oración tiene dimensión:

$$
v_{\text{sent}} \in \mathbb{R}^{2m}
$$

y cada componente queda definida como el valor máximo alcanzado en esa dimensión a lo largo de toda la secuencia.

###### **Intuición del max-pooling**

La idea es que algunas dimensiones del espacio oculto actúan como detectores de ciertos patrones semánticos o sintácticos.

Entonces, si una dimensión se activa fuertemente en alguna posición de la oración, el max-pooling la conserva.

Esto permite que el embedding final capture "las señales más importantes" distribuidas a lo largo de la secuencia.

##### **Ejemplo numérico simple**

Supongamos que una oración de longitud 3 produce los siguientes estados ocultos concatenados:

$$
h_1=
\begin{bmatrix}
0.2\\
0.7\\
-0.1\\
0.5
\end{bmatrix},
\qquad
h_2=
\begin{bmatrix}
0.8\\
0.1\\
0.4\\
0.3
\end{bmatrix},
\qquad
h_3=
\begin{bmatrix}
0.5\\
0.4\\
0.9\\
0.2
\end{bmatrix}
$$

Entonces el max-pooling por dimensión da:

- primera dimensión:
  $$
  \max(0.2,0.8,0.5)=0.8
  $$
- segunda dimensión:
  $$
  \max(0.7,0.1,0.4)=0.7
  $$
- tercera dimensión:
  $$
  \max(-0.1,0.4,0.9)=0.9
  $$
- cuarta dimensión:
  $$
  \max(0.5,0.3,0.2)=0.5
  $$

Por tanto:

$$
v_{\text{sent}}=
\begin{bmatrix}
0.8\\
0.7\\
0.9\\
0.5
\end{bmatrix}
$$

Ese sería el embedding final de la oración.

##### **Ventaja frente al promedio**

A diferencia del promedio de embeddings:

$$
\frac{1}{n}\sum_i v_i
$$

BiLSTM + max-pooling sí puede capturar:

- orden,
- contexto local y global,
- dependencias largas,
- desambiguación contextual.

Por ejemplo, en una oración larga, una palabra importante puede activar una dimensión solo en una posición concreta. El max-pooling conserva esa activación, mientras que el promedio podría diluirla.

##### **Relación con InferSent**

Este esquema se hizo especialmente conocido con modelos como **InferSent**, donde:

1. una BiLSTM produce estados contextuales,
2. max-pooling resume toda la oración,
3. el embedding resultante se usa en tareas de inferencia textual, similitud o transferencia a otros problemas.

La idea central es que una buena representación de oración debe capturar no solo qué palabras aparecen, sino cómo se organizan en contexto.

##### **Interpretación geométrica**

Puede verse así:

- cada posición $t$ produce un vector contextual $h_t$,
- el conjunto $\{h_1,\dots,h_n\}$ describe la oración a lo largo del tiempo,
- max-pooling selecciona las activaciones más fuertes en cada dirección del espacio.

Así, $v_{\text{sent}}$ funciona como un resumen compacto de las propiedades más salientes de toda la oración.

##### **Limitaciones**

Aunque BiLSTM + max-pooling es mucho más expresivo que el promedio simple, todavía tiene limitaciones:

- comprime toda la oración en un solo vector fijo,
- puede perder parte de la estructura fina,
- el max-pooling ignora en qué posición apareció una activación,
- no modela tan directamente relaciones arbitrarias entre posiciones distantes como lo hace la autoatención.

Por eso, más adelante se popularizaron modelos basados en **atención** y **transformers**.

##### **Importancia conceptual**

Este tipo de representación es importante porque marca una transición clara:

- de vectores de palabras estáticos,
- a representaciones **contextualizadas** de secuencias,
- y de allí a embeddings de oración útiles para tareas más complejas.

Conceptualmente, muestra que una oración puede representarse como el resultado de:

1. codificar cada palabra en contexto,
2. y luego resumir toda la secuencia mediante una operación global como max-pooling.

##### **5.2 Atención y embeddings contextuales**

Los modelos basados en **atención** producen representaciones **contextualizadas**, es decir, el vector de una palabra ya no depende únicamente de su identidad léxica, sino también de las demás palabras que la rodean.

Esto resuelve una limitación importante de los embeddings estáticos, como Word2Vec o GloVe:

- en esos modelos, una palabra como *banco* tiene siempre el mismo vector,
- en modelos con atención, *banco* puede tener una representación distinta en:
  - *banco central*,
  - *banco del río*.
    
##### **Idea general de atención**

La atención permite que cada token de la secuencia "mire" a otros tokens y combine información relevante de ellos.

La ecuación estándar es:

$$
\mathrm{Attention}(Q,K,V)=
\mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V.
$$

Esta ecuación es correcta y resume tres pasos:

1. medir qué tan relevante es cada token para otro,
2. convertir esas relevancias en pesos normalizados,
3. usar esos pesos para combinar información.


###### **¿Qué son $Q$, $K$ y $V$?**

Los tres objetos centrales son:

- **Query (Q)**: representa lo que un token "busca" en los demás,
- **Key (K)**: representa qué información ofrece cada token para ser comparada,
- **Value (V)**: representa la información que finalmente se va a combinar.

Si una secuencia tiene $n$ tokens y la dimensión interna es $d_k$, normalmente se construyen matrices:

$$
Q \in \mathbb{R}^{n \times d_k}, \qquad
K \in \mathbb{R}^{n \times d_k}, \qquad
V \in \mathbb{R}^{n \times d_v}
$$

a partir de los embeddings de entrada $X$:

$$
Q = XW_Q,\qquad
K = XW_K,\qquad
V = XW_V
$$

donde:

- $X \in \mathbb{R}^{n\times d}$ contiene la secuencia de embeddings de entrada,
- $W_Q, W_K, W_V$ son matrices aprendidas.

Así, cada token genera:

- una query,
- una key,
- y un value.

##### **Paso 1. Producto $QK^\top$: compatibilidad entre tokens**

El término:

$$
QK^\top
$$

calcula la compatibilidad entre cada query y cada key.

Si la secuencia tiene $n$ tokens, entonces:

$$
QK^\top \in \mathbb{R}^{n\times n}
$$

y la entrada $(i,j)$ de esta matriz es:

$$
q_i^\top k_j
$$

Esto mide cuánto debe atender el token $i$ al token $j$.

En otras palabras:

- la fila $i$ indica a qué otros tokens atiende el token $i$,
- la columna $j$ indica cuánto influye el token $j$ sobre los demás.

##### **Paso 2. Escalamiento por $\sqrt{d_k}$**

La ecuación divide por:

$$
\sqrt{d_k}
$$

quedando:

$$
\frac{QK^\top}{\sqrt{d_k}}
$$

Esto se hace para estabilizar las magnitudes de los productos internos.

Si $d_k$ es grande, el producto $q_i^\top k_j$ puede crecer mucho y empujar a la softmax hacia valores extremos.  
El escalamiento evita que las probabilidades se saturen demasiado pronto y facilita el entrenamiento.


##### **Paso 3. Softmax: pesos de atención**

Luego se aplica softmax fila por fila:

$$
A=
\mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)
$$

donde:

$$
A \in \mathbb{R}^{n\times n}
$$

La matriz $A$ contiene los **pesos de atención**.

Cada fila de $A$:

- suma 1,
- indica cómo reparte su atención un token entre todos los tokens de la secuencia.

##### **Paso 4. Combinación con $V$**

Finalmente, esos pesos se aplican a los *values*:

$$
\mathrm{Attention}(Q,K,V)=AV
$$

o de forma expandida:

$$
\mathrm{Attention}(Q,K,V)=
\mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

Esto significa que la nueva representación de cada token es una combinación ponderada de los values de toda la secuencia.

Para el token $i$, la salida es:

$$
o_i = \sum_{j=1}^{n} \alpha_{ij} v_j
$$

donde:

- $\alpha_{ij}$ es el peso de atención del token $i$ hacia el token $j$,
- $v_j$ es el value del token $j$.

##### **Interpretación intuitiva**

Cada token produce una representación nueva mirando a los demás tokens.

Así, la representación final de una palabra depende de:

- su embedding inicial,
- su posición,
- y las palabras con las que interactúa en el contexto.

Por eso se dice que el embedding es **contextualizado**.

##### **Ejemplo intuitivo**

Consideremos dos oraciones:

1. *Fui al banco a retirar dinero.*  
2. *Nos sentamos en la orilla del banco del río.*

En un modelo estático, la palabra *banco* tendría el mismo vector en ambos casos.

En un modelo con atención:

- en la primera oración, *banco* atenderá a palabras como *retirar* y *dinero*,
- en la segunda, *banco* atenderá a *orilla* y *río*.

Como resultado, el vector final de *banco* será distinto en cada oración.

##### **Ejemplo numérico simple**

Supongamos que tenemos una secuencia de 2 tokens y la matriz de puntuaciones ya escaladas es:

$$
\frac{QK^\top}{\sqrt{d_k}}=
\begin{bmatrix}
2 & 1\\
0 & 3
\end{bmatrix}
$$

Aplicando softmax por filas obtenemos aproximadamente:

$$
A=
\begin{bmatrix}
0.731 & 0.269\\
0.047 & 0.953
\end{bmatrix}
$$

Supongamos además que:

$$
V=
\begin{bmatrix}
1 & 0\\
0 & 2
\end{bmatrix}
$$

Entonces la salida es:

$$
AV=
\begin{bmatrix}
0.731 & 0.269\\
0.047 & 0.953
\end{bmatrix}
\begin{bmatrix}
1 & 0\\
0 & 2
\end{bmatrix}
=
\begin{bmatrix}
0.731 & 0.538\\
0.047 & 1.906
\end{bmatrix}
$$

Interpretación:

- el primer token combina información de ambos valores, pero se apoya más en el primero,
- el segundo token atiende casi por completo al segundo.

##### **Self-attention**

Cuando $Q$, $K$ y $V$ se construyen a partir de la **misma secuencia de entrada**, se habla de **self-attention**.

Eso ocurre en transformers como BERT.

La secuencia se representa como:

$$
X = [x_1,\dots,x_n]
$$

y de ahí se obtienen:

$$
Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V
$$

Así, cada token puede interactuar con todos los demás tokens de la misma oración.

##### **Embeddings contextuales**

La gran diferencia con embeddings clásicos es:

- **embeddings estáticos**: una palabra tiene un solo vector fijo,
- **embeddings contextuales**: una palabra produce un vector distinto según el contexto.

En notación conceptual:

- estático:
  $$
  e_w
  $$
- contextual:
  $$
  h_t = f(w_t, w_1,\dots,w_n)
  $$

Es decir, la representación de la palabra en la posición $t$ depende de toda la secuencia.

##### **Representaciones de oración en modelos tipo BERT**

Una vez que el encoder produce salidas contextualizadas:

$$
H=
\begin{bmatrix}
h_1\\
h_2\\
\vdots\\
h_n
\end{bmatrix}
\in \mathbb{R}^{n\times d}
$$

hay varias formas de obtener una representación de toda la oración.

Dos prácticas comunes son:

##### **Opción 1. Usar el token especial $[CLS]$**

En modelos tipo BERT, se añade al inicio de la secuencia un token especial:

$$
[\mathrm{CLS}], w_1,\dots,w_n
$$

Después del encoder, la salida correspondiente a $[CLS]$ puede usarse como embedding de la oración:

$$
v_{\text{sent}} = h_{\mathrm{[CLS]}}
$$

La idea es que ese token aprende a concentrar información global útil para tareas posteriores.

##### **Opción 2. Mean pooling**

Otra estrategia es promediar las salidas contextualizadas de todos los tokens:

$$
v_{\text{sent}} = \frac{1}{n}\sum_{t=1}^{n} h_t
$$

A veces se excluyen tokens especiales o padding, pero la idea básica es la misma: construir una representación global mediante el promedio de las salidas del encoder.

###### **Importante: no hay una única estrategia universal**

Ni $[CLS]$ ni *mean pooling* deben presentarse como la única solución correcta.

Su utilidad depende de:

- la arquitectura,
- el tipo de entrenamiento previo,
- la tarea downstream,
- y el comportamiento empírico del modelo.

En algunos casos $[CLS]$ funciona muy bien, sobre todo si el modelo fue entrenado para usarlo en clasificación.  
En otros casos, el *mean pooling* produce embeddings más estables o más útiles para similitud semántica y recuperación.

###### **Ventajas de la atención frente a BiLSTM**

La atención tiene varias ventajas conceptuales:

- permite interacción directa entre cualquier par de posiciones,
- captura dependencias largas sin pasar secuencialmente por todos los estados,
- es altamente paralelizable,
- facilita representaciones contextuales profundas.

Por eso los transformers reemplazaron en gran medida a muchas arquitecturas recurrentes en NLP moderno.

##### **Limitaciones y observaciones**

Aunque la ecuación de atención es elegante, en la práctica hay varios detalles adicionales:

- **multi-head attention**: se aplican varias atenciones en paralelo,
- **positional encodings**: se añade información de orden, porque la atención por sí sola no impone una secuencia,
- **capas feed-forward**: después de la atención se añaden transformaciones no lineales,
- **máscaras**: se usan en algunos modelos para restringir atención a ciertas posiciones.

Por eso, la ecuación de atención es el núcleo del mecanismo, pero no describe por sí sola toda la arquitectura transformer.

###### **Importancia conceptual**

La atención y los embeddings contextuales marcan un cambio decisivo en NLP:

- se pasa de representar palabras como objetos aislados,
- a representarlas como funciones del contexto completo.

Ese cambio fue fundamental para:

- BERT,
- modelos encoder-decoder,
- LLM,
- y más adelante, arquitecturas multimodales donde texto e imagen interactúan en espacios compartidos o mediante atención cruzada.

#### **6. Embeddings estáticos vs. contextuales**

- **Estáticos**: una palabra tiene el mismo vector siempre (`Word2Vec`, `GloVe`, `FastText`).
- **Contextuales**: el vector depende del entorno lingüístico (`ELMo`, `BERT`, `T5`, modelos encoder-decoder y LLM modernos).

Para un curso orientado a multimodalidad e IA generativa, esta distinción es clave:
- Los primeros modelos multimodales trabajaban sobre **representaciones estáticas** o agregadas,
- Los modelos modernos integran codificadores contextuales y después los acoplan con visión, audio o video.

#### **7. Conexión con aprendizaje multimodal**

Las representaciones distribuidas de palabras, frases y oraciones no son solo un tema de NLP: constituyen la base conceptual de muchos de los primeros modelos de **alineamiento visual-semántico**.

La idea central es la siguiente:

- el texto debe representarse como un vector denso en un espacio continuo,
- la imagen también debe representarse como un vector denso,
- ambos vectores deben proyectarse a un **espacio compartido** donde sea posible compararlos directamente.

Esto explica por qué estudiar embeddings, composicionalidad y representaciones contextualizadas es pertinente antes de entrar a modelos multimodales tempranos: sin buenas representaciones textuales y visuales, no hay alineamiento útil entre modalidades.


##### **Representación textual y visual**

Supongamos que:

- $x$ es una instancia textual, por ejemplo una palabra, una frase o una oración
- $I$ es una imagen
- $t$ es la representación vectorial del texto
- $v$ es la representación vectorial de la imagen.

Formalmente:

$$
t \in \mathbb{R}^{d_t}, \qquad v \in \mathbb{R}^{d_v}
$$

Aquí:

- $t$ puede provenir de un promedio de embeddings, una BiLSTM, un encoder con atención, etc.
- $v$ puede provenir de descriptores visuales, CNN features, region features o embeddings visuales más modernos.

Como ambas modalidades suelen vivir inicialmente en espacios distintos, no es conveniente compararlas directamente.

###### **Proyección a un espacio compartido**

Una forma elemental de alinear texto e imagen es aprender dos transformaciones lineales:

$$
z_t = W_t\, t,\qquad z_v = W_v\, v
$$

donde:

- $W_t \in \mathbb{R}^{d \times d_t}$ proyecta el texto,
- $W_v \in \mathbb{R}^{d \times d_v}$ proyecta la imagen,
- $z_t, z_v \in \mathbb{R}^{d}$ son las representaciones proyectadas en el mismo espacio compartido.

La idea es que, después de la proyección:

- una imagen y su descripción correcta queden cerca,
- una imagen y un texto no relacionado queden lejos.

###### **Interpretación geométrica**

Antes de proyectar:

- el texto vive en un espacio semántico textual,
- la imagen vive en un espacio visual.

Después de aplicar $W_t$ y $W_v$, ambos deben quedar en un espacio común donde "cerca" signifique compatibilidad semántica entre modalidades.

Por ejemplo:

- la imagen de un perro y la frase *"un perro corriendo"* deberían quedar próximas,
- la misma imagen y la frase *"una torre de concreto"* deberían quedar alejadas.

###### **Similitud en el espacio compartido**

Una vez obtenidos $z_t$ y $z_v$, la compatibilidad entre imagen y texto puede medirse con una función de similitud.

Dos opciones comunes son:

##### **Producto interno**
$$
s(I,x)= z_v^\top z_t
$$

##### **Similitud coseno**
$$
s(I,x)= \cos(z_v,z_t)
=
\frac{z_v^\top z_t}{\|z_v\|\,\|z_t\|}
$$

###### **¿Qué diferencia hay entre ambas?**

**Producto interno**
- depende tanto del ángulo como de la magnitud de los vectores,
- puede ser útil si las normas de los vectores contienen información relevante.

**Similitud coseno**
- depende solo del ángulo entre vectores,
- ignora la magnitud,
- suele ser preferida cuando interesa comparar dirección semántica más que escala.

En muchos modelos tempranos de alineamiento, cualquiera de las dos puede aparecer como función de compatibilidad.

###### **Ejemplo numérico simple**

Supongamos que en el espacio compartido tenemos:

$$
z_v =
\begin{bmatrix}
1\\
2
\end{bmatrix},
\qquad
z_t =
\begin{bmatrix}
2\\
1
\end{bmatrix}
$$

Entonces el producto interno es:

$$
z_v^\top z_t = 1\cdot 2 + 2\cdot 1 = 4
$$

Las normas son:

$$
\|z_v\| = \sqrt{1^2+2^2}=\sqrt{5},
\qquad
\|z_t\| = \sqrt{2^2+1^2}=\sqrt{5}
$$

y la similitud coseno es:

$$
\cos(z_v,z_t)=\frac{4}{\sqrt{5}\sqrt{5}}=\frac{4}{5}=0.8
$$

Un valor de 0.8 indica que ambos vectores están bastante alineados.

###### **Objetivo de entrenamiento: pares positivos y negativos**

No basta con definir una similitud, el modelo debe aprender a asignar:

- alta similitud a pares correctos,
- baja similitud a pares incorrectos.

Supongamos:

- $x^+$: texto correcto asociado a la imagen $I$,
- $x^-$: texto incorrecto o no correspondiente.

Entonces queremos que:

$$
s(I,x^+) > s(I,x^-)
$$

y, más aún, que la diferencia sea al menos un margen $\alpha > 0$.

###### **Pérdida de margen**

Una formulación clásica es:

$$
\mathcal{L}=\max\bigl(0,\alpha - s(I,x^+) + s(I,x^-)\bigr)
$$

Esta ecuación puede entenderse así:

- si el par positivo ya puntúa al menos $\alpha$ más que el negativo, la pérdida es cero,
- si no, el modelo es penalizado.


###### **Interpretación del término interno**

El término dentro del máximo es:

$$
\alpha - s(I,x^+) + s(I,x^-)
$$

Queremos que sea menor o igual que 0, es decir:

$$
s(I,x^+) - s(I,x^-) \ge \alpha
$$

Esto significa:

- el texto correcto debe estar más cerca de la imagen que el texto incorrecto,
- y esa diferencia debe superar un margen mínimo.

###### **Ejemplo numérico**

Supongamos:

$$
\alpha = 0.2
$$

y que el modelo asigna:

$$
s(I,x^+) = 0.9,
\qquad
s(I,x^-) = 0.4
$$

Entonces:

$$
\alpha - s(I,x^+) + s(I,x^-)
=
0.2 - 0.9 + 0.4
=
-0.3
$$

Por tanto:

$$
\mathcal{L}=\max(0,-0.3)=0
$$

No hay penalización, porque el positivo ya supera al negativo por más del margen.

Ahora supongamos:

$$
s(I,x^+) = 0.55,
\qquad
s(I,x^-) = 0.45
$$

Entonces:

$$
0.2 - 0.55 + 0.45 = 0.10
$$

y la pérdida es:

$$
\mathcal{L}=\max(0,0.10)=0.10
$$

Aquí el modelo sí es penalizado, porque la separación entre positivo y negativo todavía es insuficiente.

###### **Qué aprende realmente el modelo**

Los parámetros $W_t$ y $W_v$ se ajustan para que:

- textos correctos e imágenes correctas se acerquen en el espacio compartido,
- pares incorrectos se separen.

Si además el encoder textual y/o visual es entrenable, entonces también se refinan las representaciones iniciales $t$ y $v$.

En conjunto, el sistema aprende:

1. una geometría compartida entre modalidades,
2. una noción de compatibilidad semántica imagen-texto.

##### **Relación con modelos tempranos de alineamiento visual-semántico**

Este esquema es conceptualmente muy cercano a muchos modelos tempranos de la literatura:

- **visual-semantic embeddings**,
- **cross-modal embeddings**,
- **image-sentence retrieval**,
- **ranking-based alignment**.

La estructura general suele ser:

1. extraer una representación textual,
2. extraer una representación visual,
3. proyectarlas a un espacio común,
4. optimizar una similitud con una pérdida de ranking o margen.

##### **Por qué esto conecta con representaciones distribuidas**

Este puente explica por qué estudiar representaciones distribuidas sí es pertinente antes de entrar a multimodalidad:

- si el texto estuviera representado solo por one-hot vectors, sería difícil capturar semántica,
- si no hubiera embeddings composicionales para frases u oraciones, sería difícil alinear descripciones completas con imágenes,
- si no entendiéramos similitud vectorial, producto interno o coseno, no podríamos interpretar el objetivo multimodal.

En otras palabras:

> el alineamiento multimodal temprano hereda directamente la lógica de los espacios vectoriales distribuidos.

###### **Conexión importante**

Estudiar representaciones distribuidas en esta etapa tiene sentido porque prepara los conceptos que luego aparecen en modelos tempranos de aprendizaje multimodal:

- espacios semánticos compartidos,
- compatibilidad entre modalidades,
- embeddings de texto y de imagen,
- pérdidas de ranking,
- y alineamiento visual-semántico.

Así, la transición natural es:

$$
\text{embeddings de palabras}
\rightarrow
\text{frases y oraciones}
\rightarrow
\text{proyección compartida texto-imagen}
\rightarrow
\text{alineamiento multimodal temprano}
$$

Por eso, entender bien las representaciones distribuidas no es un desvío: es una base necesaria para comprender cómo surgieron los primeros modelos multimodales.

#### **8. Ejercicios sugeridos**

1. Explica por qué el coseno entre embeddings resulta más útil que la distancia euclídea en muchos espacios semánticos.
2. Compara NNLM y C&W: ¿qué gana cada uno con su función objetivo?
3. Implementa una variante de CBOW que use suma en lugar de promedio del contexto.
4. Toma una frase y compara:
   - promedio simple,
   - promedio ponderado,
   - una representación contextual descrita conceptualmente.
5. Explica por qué un modelo subpalabra puede ayudar en dominios con neologismos o errores ortográficos.

#### **9. Cierre**

Este cuaderno deja una base suficientemente sólida para entrar a:
- modelos tempranos de alineamiento visual-semántico
- semántica distribucional multimodal
- variantes neuronales tempranas
- y la transición posterior hacia embeddings contextuales, LLM y modelos multimodales más avanzados.